# L5 — Raggruppare, riassumere, disegnare (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/L05_raggruppare/L05_soluzioni.ipynb)

## Contesto

Il dataset è pulito (è il risultato della L3). Ora possiamo finalmente fare domande da economisti: come va il fatturato nel tempo? Quale regione ha le imprese più grandi? Quale settore è più produttivo?

**La domanda di oggi:** il numero che ottieni racconta davvero quello che sembra?

## 0. Preparazione

Esegui questa cella così com'è: carica il dataset pulito e la libreria per i grafici.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/imprese_pulito.xlsx")
df.shape

*Solo nelle soluzioni.* La cella qui sotto cambia l'aspetto di **tutti** i grafici che seguono: dimensione, griglia leggera, niente bordi superiore e destro, titoli in grassetto. Non è obbligatoria: i grafici funzionano anche senza.

In [ ]:
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.prop_cycle": plt.cycler(color=["#2a6f97", "#e07a5f", "#81b29a", "#f2cc8f",
                                         "#6d597a", "#3d405b", "#b56576"]),
})

## Parte 1 — Nuove colonne

### 1. La produttività

Crea una nuova colonna `produttivita`: fatturato diviso addetti (migliaia di euro per addetto). Poi mostra le prime 5 righe per controllare.

🔎 *Da cercare:* `pandas nuova colonna` · `pandas dividere colonne`

In [ ]:
df["produttivita"] = df["fatturato"] / df["addetti"]
df[["fatturato", "addetti", "produttivita"]].head()

### 2. Controllala

Quanti valori mancanti ha `produttivita`? Sono di più di quelli di `fatturato`? Perché?

🔎 *Da cercare:* `pandas isna sum`

In [ ]:
print(df["fatturato"].isna().sum(), df["addetti"].isna().sum(), df["produttivita"].isna().sum())

*Risposta:* Sono di più: basta che manchi **uno** dei due valori (fatturato o addetti) perché il risultato sia mancante.

## Parte 2 — Raggruppare

### 3. Fatturato medio per anno

Calcola il fatturato **medio** per ogni anno.

🔎 *Da cercare:* `pandas groupby mean`

In [ ]:
df.groupby("anno")["fatturato"].mean()

### 4. Più statistiche insieme

Per ogni anno calcola media, mediana e numero di osservazioni del fatturato, in un'unica tabella.

🔎 *Da cercare:* `pandas groupby agg`

In [ ]:
df.groupby("anno")["fatturato"].agg(["mean", "median", "count"])

### 5. La regione più ricca?

Calcola il fatturato medio per regione e ordinalo dal più alto al più basso. Quale regione è prima?

🔎 *Da cercare:* `pandas sort_values ascending`

In [ ]:
df.groupby("regione")["fatturato"].mean().sort_values(ascending=False)

### 6. Ne sei sicuro?

Rifai la tabella per regione con **media**, **mediana** e **numero di imprese diverse**. La classifica regge? Guarda in particolare le prime due regioni.

🔎 *Da cercare:* `pandas agg named aggregation` · `pandas nunique`

In [ ]:
df.groupby("regione").agg(
    media=("fatturato", "mean"),
    mediana=("fatturato", "median"),
    imprese=("id_impresa", "nunique"),
).sort_values("media", ascending=False).head()

*Risposta:* La Liguria è prima per media solo grazie a un'impresa enorme (la sua mediana è nella norma). La Basilicata ha la mediana più alta, ma solo 2 imprese: troppo poche per dire qualcosa. Media, mediana e numero di osservazioni vanno guardati insieme.

### 7. I settori

Calcola la produttività **mediana** per settore, ordinata. Quale settore è il più produttivo?

🔎 *Da cercare:* `pandas groupby median`

In [ ]:
df.groupby("settore")["produttivita"].median().sort_values()

## Parte 3 — Disegnare

### 8. Nel tempo

Disegna un grafico a **linea** del fatturato **mediano** per anno. Cosa succede nel 2020?

🔎 *Da cercare:* `pandas plot line`

In [ ]:
df.groupby("anno")["fatturato"].median().plot(marker="o")
plt.xticks(sorted(df["anno"].unique()))
plt.ylim(bottom=0)
plt.title("Fatturato mediano delle imprese, 2019-2023")
plt.xlabel("Anno")
plt.ylabel("Migliaia di euro")
plt.show()

*Risposta:* Nel 2020 il fatturato mediano crolla (circa un quarto in meno) e nel 2021 torna ai livelli precedenti: è lo shock della pandemia.

### 9. Per categoria

Disegna un grafico a **barre orizzontali** della produttività mediana per settore, ordinate.

🔎 *Da cercare:* `pandas plot barh`

In [ ]:
df.groupby("settore")["produttivita"].median().sort_values().plot(kind="barh")
plt.title("Produttività mediana per settore")
plt.xlabel("Migliaia di euro per addetto")
plt.ylabel("")
plt.grid(axis="y", visible=False)
plt.show()

### 10. La distribuzione

Disegna un **istogramma** del fatturato. Che forma ha? Ti spiega perché media e mediana sono così diverse?

🔎 *Da cercare:* `pandas plot hist bins`

In [ ]:
df["fatturato"].plot(kind="hist", bins=50, edgecolor="white")
plt.axvline(df["fatturato"].median(), color="#e07a5f", linestyle="--", label="mediana")
plt.axvline(df["fatturato"].mean(), color="#3d405b", linestyle="--", label="media")
plt.legend(handles=plt.gca().lines)   # in legenda solo le due linee
plt.title("Distribuzione del fatturato delle imprese")
plt.xlabel("Fatturato (migliaia di euro)")
plt.ylabel("Numero di osservazioni")
plt.show()
print(df["fatturato"].mean(), df["fatturato"].median())

*Risposta:* È molto asimmetrica: tantissime imprese piccole e poche molto grandi. Le poche grandi tirano su la media (circa 2.600) ben sopra la mediana (circa 1.400).

## Domanda finale

Un giornalista ti chiede: *"Qual è la regione con le imprese più ricche?"* Scrivi una frase che gli diresti, e una che **non** gli diresti.

*Risposta:* Direi: "le differenze tra regioni sono piccole; per fatturato mediano spiccano Umbria e Sardegna, ma su pochi dati". Non direi: "la Liguria è la regione più ricca" (lo è solo in media, per un'impresa sola) né "la Basilicata" (2 imprese).

---
## ⭐ Bonus (per chi ha finito)

### B1. Un grafico presentabile

Riprendi il grafico nel tempo e aggiungi titolo, etichette degli assi con le unità di misura, e salvalo come immagine `fatturato_anno.png`.

🔎 *Da cercare:* `matplotlib title xlabel ylabel` · `matplotlib savefig`

In [ ]:
df.groupby("anno")["fatturato"].median().plot(marker="o")
plt.xticks(sorted(df["anno"].unique()))
plt.ylim(bottom=0)
plt.title("Fatturato mediano delle imprese, 2019-2023")
plt.xlabel("Anno")
plt.ylabel("Migliaia di euro")
plt.savefig("fatturato_anno.png", dpi=150, bbox_inches="tight")
plt.show()

### B2. Una tabella a doppia entrata

Costruisci una tabella con le regioni sulle righe, gli anni sulle colonne e il fatturato mediano nelle celle.

🔎 *Da cercare:* `pandas pivot_table`

In [ ]:
df.pivot_table(index="regione", columns="anno", values="fatturato", aggfunc="median").round(0)

### B3. Scala logaritmica

Rifai l'istogramma del fatturato con l'asse orizzontale in scala logaritmica. Cosa cambia?

🔎 *Da cercare:* `numpy log` · `matplotlib xscale log`

In [ ]:
import numpy as np

# classi (bins) larghe uguali in scala logaritmica, dal fatturato minimo al massimo
f = df["fatturato"].dropna()
f = f[f > 0]   # il logaritmo esiste solo per i numeri positivi
classi = np.logspace(np.log10(f.min()), np.log10(f.max()), 50)

f.plot(kind="hist", bins=classi, edgecolor="white")
plt.xscale("log")
plt.xticks([100, 1000, 10000], ["100", "1.000", "10.000"])
plt.axvline(f.median(), color="#e07a5f", linestyle="--", label="mediana")
plt.axvline(f.mean(), color="#3d405b", linestyle="--", label="media")
plt.legend(handles=plt.gca().lines)
plt.title("Distribuzione del fatturato, scala logaritmica")
plt.xlabel("Fatturato (migliaia di euro, scala logaritmica)")
plt.ylabel("Numero di osservazioni")
plt.show()

*Risposta:* In scala lineare quasi tutte le imprese stanno schiacciate nelle prime barre. In scala logaritmica la distribuzione diventa una campana quasi simmetrica: si leggono sia le imprese piccole sia quelle grandi. Si vede anche che la media sta a destra della mediana, tirata dalle imprese più grandi.

### B4. La crescita di ogni impresa

Il dataset è un **panel**: le stesse imprese, anno dopo anno. Calcola la crescita % del fatturato di ogni impresa rispetto all'anno prima, in una nuova colonna `crescita`. Attenzione: ogni impresa va confrontata con **se stessa**, non con la riga sopra (che per il primo anno è un'altra impresa). Poi calcola la crescita mediana per anno: quale anno è andato peggio? Perché il 2019 è vuoto?

🔎 *Da cercare:* `pandas sort_values più colonne` · `pandas groupby pct_change`

In [ ]:
df = df.sort_values(["id_impresa", "anno"])
df["crescita"] = df.groupby("id_impresa")["fatturato"].pct_change(fill_method=None) * 100
df.groupby("anno")["crescita"].median().round(1)

*Risposta:* Il 2020 (circa −24%), seguito dal rimbalzo del 2021 (+36%). Il 2019 è vuoto perché è il primo anno: non c'è un anno prima con cui confrontarlo.

---
## 📊 Per casa: altri grafici

Quattro esercizi sui grafici, da fare a casa prima della L6. Ripartono tutti da `df` con la colonna `produttivita` (Parte 1): se hai riavviato la sessione, riesegui prima la Preparazione e il passo 1.

Per ogni grafico, chiediti: **cosa si vede al primo sguardo? È vero?**

### D1. Media e mediana nello stesso grafico

Disegna in un unico grafico a linea **due linee**: il fatturato medio e quello mediano, anno per anno. Suggerimento: se `agg` restituisce una tabella con due colonne, `plot` disegna una linea per colonna, con la legenda. Aggiungi titolo e unità di misura.

1. Le due linee raccontano la stessa storia? Perché sono così lontane?
2. Guarda l'asse orizzontale: esiste l'anno 2019,5? Fai in modo che compaiano solo gli anni veri.
3. La legenda è in inglese (`mean`, `median`): fai in modo che dica `media` e `mediana`.

🔎 *Da cercare:* `pandas groupby agg` · `matplotlib xticks` · `pandas agg named aggregation`

In [ ]:
tab = df.groupby("anno")["fatturato"].agg(media="mean", mediana="median")
tab.plot(marker="o")
plt.xticks(tab.index)
plt.ylim(bottom=0)
plt.title("Fatturato medio e mediano, 2019-2023")
plt.xlabel("Anno")
plt.ylabel("Migliaia di euro")
plt.show()

*Risposta:* La storia è la stessa (crollo nel 2020, ripresa nel 2021), ma la media sta sempre molto sopra la mediana (circa 2.700 contro 1.450): la distanza tra le due linee è l'asimmetria che si vede nell'istogramma. Gli anni sono numeri, e matplotlib mette le tacche dove gli sembra meglio, anche a metà anno: `plt.xticks(tab.index)` le mette solo sugli anni della tabella. Con l'aggregazione con nomi (passo 6) le colonne, e quindi la legenda, prendono i nomi che scegli tu.

### D2. Una linea per settore

Costruisci una tabella con gli anni sulle righe, i settori sulle colonne e il fatturato **mediano** nelle celle (è la tabella a doppia entrata del bonus B2), poi disegnala: una linea per settore.

1. Tutti i settori crollano nel 2020? Quale perde di più, e quale di meno?
2. Il grafico si legge bene, con 7 linee? La legenda copre qualche linea? Spostala fuori dal grafico.

Per rispondere alla prima domanda con i numeri, calcola la variazione % tra 2019 e 2020: $\left(\frac{\text{fatturato}_{2020}}{\text{fatturato}_{2019}} - 1\right) \times 100$.

🔎 *Da cercare:* `pandas pivot_table` · `matplotlib legend bbox_to_anchor` · `pandas loc`

In [ ]:
t = df.pivot_table(index="anno", columns="settore", values="fatturato", aggfunc="median")
t.plot(marker="o", figsize=(9, 4.5))
plt.xticks(t.index)
plt.legend(title="Settore", bbox_to_anchor=(1, 1), loc="upper left", frameon=False)
plt.title("Fatturato mediano per settore, 2019-2023")
plt.xlabel("Anno")
plt.ylabel("Migliaia di euro")
plt.show()

((t.loc[2020] / t.loc[2019] - 1) * 100).round(1).sort_values()

*Risposta:* Crollano tutti. Perde di più l'alloggio e ristorazione (circa −30%), di meno i trasporti (circa −14%). Con 7 linee il grafico diventa un groviglio: si vede il 2020, ma è difficile seguire un settore. Per un confronto preciso servono i numeri, o meno linee. Con `bbox_to_anchor=(1, 1)` la legenda si aggancia all'angolo in alto a destra del grafico, fuori dagli assi.

### D3. Confrontare distribuzioni: il boxplot

Nel passo 7 hai confrontato la produttività **mediana** dei settori. Ma come sono distribuiti i valori dentro ogni settore? Disegna un **boxplot** della produttività per settore, in orizzontale.

Come si legge: la riga dentro la scatola è la mediana; la scatola va dal 25% al 75% delle imprese; i punti isolati sono valori lontani dagli altri.

Le differenze tra le mediane dei settori sono grandi o piccole, rispetto alle differenze tra le imprese dello stesso settore?

🔎 *Da cercare:* `pandas boxplot by` · `pandas boxplot vert`

In [ ]:
df.boxplot(column="produttivita", by="settore", vert=False)
plt.title("Produttività per settore")
plt.suptitle("")
plt.xlabel("Migliaia di euro per addetto")
plt.ylabel("")
plt.show()

*Risposta:* Piccole: le mediane vanno da circa 101 (servizi ICT) a 128 (manifattura), mentre dentro ogni settore metà delle imprese sta tra circa 85 e 150. Le scatole si sovrappongono quasi del tutto. La classifica del passo 7 è vera, ma dice poco su una singola impresa. (`plt.suptitle("")` toglie il titolo automatico che pandas aggiunge sopra il grafico.)

### D4. Un grafico che inganna

Disegna di nuovo le barre orizzontali della produttività mediana per settore (passo 9), poi aggiungi `plt.xlim(100, 130)`: l'asse non parte più da zero.

1. Guardando solo il grafico, quante volte la manifattura sembra più produttiva dei servizi ICT?
2. Quante volte lo è davvero? Calcolalo.

🔎 *Da cercare:* `matplotlib xlim`

In [ ]:
m = df.groupby("settore")["produttivita"].median().sort_values()
m.plot(kind="barh")
plt.xlim(100, 130)
plt.title("Produttività mediana per settore (asse tagliato)")
plt.xlabel("Migliaia di euro per addetto")
plt.ylabel("")
plt.grid(axis="y", visible=False)
plt.show()

print(round(m["Manifattura"] / m["Servizi ICT"], 2))

*Risposta:* Nel grafico tagliato la barra della manifattura è circa 25 volte quella dei servizi ICT. In realtà la manifattura è più produttiva di circa il 27% ($128{,}1 / 101{,}1 \approx 1{,}27$). Con le barre conta la **lunghezza**, quindi l'asse deve partire da zero. Con le linee no: lì conta l'andamento, e un asse che parte da zero può schiacciare tutto.

---
## Compito a casa

- Gli esercizi D1–D4 qui sopra (~30 min)
- [Kaggle Learn – Pandas](https://www.kaggle.com/learn/pandas), lezione **Grouping and Sorting** (~20 min)
- Facoltativo: [QuantEcon – Matplotlib](https://python-programming.quantecon.org/matplotlib.html)